In [1]:
import pandas as pd


df = pd.read_parquet('../Imdb_Movie_Dataset.parquet')
df_aux = df.copy()

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
import xgboost as xgb
import numpy as np
import pandas as pd
import pickle
import gzip
import gc
import shap

df_aux['runtime'] = pd.to_numeric(df_aux['runtime'], errors='coerce')
df_aux['budget'] = pd.to_numeric(df_aux['budget'], errors='coerce')
df_aux['vote_average'] = pd.to_numeric(df_aux['vote_average'], errors='coerce')
df_aux['vote_count'] = pd.to_numeric(df_aux['vote_count'], errors='coerce')

df_with_runtime = df_aux[
    df_aux['runtime'].notna() & 
    (df_aux['runtime'] >= 2) & 
    (df_aux['runtime'] <= 250)
].copy()

cpi_history = {
    1913: 9.9,   1914: 10.0,  1915: 10.1,  1916: 10.9,  1917: 12.8,  1918: 15.1,  1919: 17.3,
    1920: 20.0,  1921: 17.9,  1922: 16.8,  1923: 17.1,  1924: 17.1,  1925: 17.5,  1926: 17.7,
    1927: 17.4,  1928: 17.1,  1929: 17.1,  1930: 16.7,  1931: 15.2,  1932: 13.7,  1933: 13.0,
    1934: 13.4,  1935: 13.7,  1936: 13.9,  1937: 14.4,  1938: 14.1,  1939: 13.9,  1940: 14.0,
    1941: 14.7,  1942: 16.3,  1943: 17.3,  1944: 17.6,  1945: 18.0,  1946: 19.5,  1947: 22.3,
    1948: 24.1,  1949: 23.8,  1950: 24.1,  1951: 26.0,  1952: 26.5,  1953: 26.7,  1954: 26.9,
    1955: 26.8,  1956: 27.2,  1957: 28.1,  1958: 28.9,  1959: 29.1,  1960: 29.6,  1961: 29.9,
    1962: 30.2,  1963: 30.6,  1964: 31.0,  1965: 31.5,  1966: 32.4,  1967: 33.4,  1968: 34.8,
    1969: 36.7,  1970: 38.8,  1971: 40.5,  1972: 41.8,  1973: 44.4,  1974: 49.3,  1975: 53.8,
    1976: 56.9,  1977: 60.6,  1978: 65.2,  1979: 72.6,  1980: 82.4,  1981: 90.9,  1982: 96.5,
    1983: 99.6,  1984: 103.9, 1985: 107.6, 1986: 109.6, 1987: 113.6, 1988: 118.3, 1989: 124.0,
    1990: 130.7, 1991: 136.2, 1992: 140.3, 1993: 144.5, 1994: 148.2, 1995: 152.4, 1996: 156.9,
    1997: 160.5, 1998: 163.0, 1999: 166.6, 2000: 172.2, 2001: 177.1, 2002: 179.9, 2003: 184.0,
    2004: 188.9, 2005: 195.3, 2006: 201.6, 2007: 207.34, 2008: 215.30, 2009: 214.54, 2010: 218.06,
    2011: 224.94, 2012: 229.59, 2013: 232.96, 2014: 236.74, 2015: 237.02, 2016: 240.01, 2017: 245.12,
    2018: 251.11, 2019: 255.66, 2020: 258.81, 2021: 270.97, 2022: 292.66, 2023: 304.70, 2024: 313.20,
    2025: 320.10, 2026: 326.50
}

cpi_2026 = cpi_history[2026]

def ajustar_orcamento(row):
    ano = row['release_year']
    orcamento = row['budget']
    if orcamento <= 0:
        return 0
    cpi_ano = cpi_history.get(ano)
    if not cpi_ano:
        ano_proximo = min(cpi_history.keys(), key=lambda x: abs(x - ano))
        cpi_ano = cpi_history[ano_proximo]
    multiplicador = cpi_2026 / cpi_ano
    return orcamento * multiplicador

df_with_runtime['release_year'] = pd.to_datetime(df_with_runtime['release_date'], errors='coerce').dt.year
median_year = df_with_runtime['release_year'].median()
df_with_runtime['release_year'] = df_with_runtime['release_year'].fillna(median_year).astype(int)

df_with_runtime['release_5_years'] = (df_with_runtime['release_year'] // 5) * 5

df_with_runtime['movie_age'] = 2027 - df_with_runtime['release_year']

df_with_runtime['has_budget'] = (df_with_runtime['budget'] > 0).astype('int8')
df_with_runtime['budget'] = df_with_runtime['budget'].fillna(0)
df_with_runtime['budget'] = df_with_runtime.apply(ajustar_orcamento, axis=1)

df_with_runtime['overview_len'] = df_with_runtime['overview'].astype(str).fillna('').str.len()
df_with_runtime['tagline_len'] = df_with_runtime['tagline'].astype(str).fillna('').str.len()

df_with_runtime['keywords'] = df_with_runtime['keywords'].astype(str).fillna('')
df_with_runtime['is_short_keyword'] = df_with_runtime['keywords'].str.contains('short', case=False, regex=False).astype('int8')

df_with_runtime['vote_average'] = df_with_runtime['vote_average'].fillna(df_with_runtime['vote_average'].median())
df_with_runtime['vote_count'] = df_with_runtime['vote_count'].fillna(0)
df_with_runtime['log_vote_count'] = np.log1p(df_with_runtime['vote_count'])
df_with_runtime['vote_score_interact'] = df_with_runtime['vote_average'] * df_with_runtime['log_vote_count']

df_with_runtime['votes_per_year'] = df_with_runtime['vote_count'] / (df_with_runtime['movie_age'] + 1)
df_with_runtime['is_en'] = (df_with_runtime['original_language'] == 'en').astype('int8')

df_with_runtime['genres'] = df_with_runtime['genres'].astype(str).fillna('')
df_with_runtime['main_genre'] = df_with_runtime['genres'].str.split(', ').str[0]

global_mean = df_with_runtime['runtime'].mean()
genre_stats = df_with_runtime.groupby('main_genre')['runtime'].agg(['mean', 'count'])

smoothing_g = 15
df_with_runtime['genre_mean'] = df_with_runtime['main_genre'].map(
    lambda x: (genre_stats.loc[x, 'mean'] * genre_stats.loc[x, 'count'] + global_mean * smoothing_g) / (genre_stats.loc[x, 'count'] + smoothing_g) if x in genre_stats.index else global_mean
)

genre_decade_stats = df_with_runtime.groupby(['main_genre', 'release_5_years'])['runtime'].agg(['mean', 'count'])
smoothing_gd = 20
def get_smooth_genre_decade(row):
    key = (row['main_genre'], row['release_5_years'])
    if key in genre_decade_stats.index:
        stat = genre_decade_stats.loc[key]
        return (stat['mean'] * stat['count'] + row['genre_mean'] * smoothing_gd) / (stat['count'] + smoothing_gd)
    return row['genre_mean']

df_with_runtime['genre_decade_mean'] = df_with_runtime.apply(get_smooth_genre_decade, axis=1)

numerical_features = [
    'vote_average', 'vote_count', 'log_vote_count', 'release_year', 'release_5_years', 'movie_age',
    'budget', 'is_short_keyword', 'overview_len', 'tagline_len', 
    'genre_mean', 'genre_decade_mean', 'vote_score_interact',
    'has_budget', 'is_en', 'votes_per_year'
]
multi_value_categorical_features = ['genres', 'production_companies', 'production_countries']
single_value_categorical_features = []

all_base_features = numerical_features + multi_value_categorical_features + single_value_categorical_features + ['main_genre']

df_train = df_with_runtime[all_base_features + ['runtime']].copy()

top_n_categories = 50
cols_to_drop = ['main_genre'] if 'main_genre' in df_train.columns else []
new_features_dict = {}

for col in multi_value_categorical_features:
    df_train[col] = df_train[col].astype(str).fillna('')
    item_counts = df_train[col].str.split(', ').explode().str.strip().value_counts()
    top_items = item_counts[item_counts.index != ''].head(top_n_categories).index.tolist()
    
    for item_name in top_items:
        new_features_dict[f'{col}_{item_name}'] = df_train[col].str.contains(item_name, regex=False, na=False).astype('int8')
    
    cols_to_drop.append(col)

df_new_features = pd.DataFrame(new_features_dict, index=df_train.index)
df_train = pd.concat([df_train, df_new_features], axis=1)
df_train.drop(columns=cols_to_drop, inplace=True, errors='ignore')

df_train = pd.get_dummies(df_train, columns=[col for col in single_value_categorical_features], drop_first=True)

df_shorts = df_train[df_train['runtime'] < 40].copy()
df_features_shorts = df_shorts.drop('runtime', axis=1)
labels_shorts = df_shorts['runtime']

df_features_longs = df_train[df_train['runtime'] >= 40].copy()
labels_longs = df_features_longs['runtime']
df_features_longs.drop('runtime', axis=1, inplace=True)

features_shorts = df_features_shorts.columns.tolist()
features_longs = df_features_longs.columns.tolist()

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(df_features_shorts, labels_shorts, test_size=0.2, random_state=42)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(df_features_longs, labels_longs, test_size=0.2, random_state=42)

del df_with_runtime, df_train, df_shorts, df_features_shorts, df_features_longs
del labels_shorts, labels_longs, df_new_features, new_features_dict
gc.collect()

model_shorts_runtime = xgb.XGBRegressor(
    n_estimators=800, learning_rate=0.02, max_depth=5, min_child_weight=3,
    subsample=0.8, colsample_bytree=0.7, objective='reg:squarederror',
    random_state=42, n_jobs=-1
)
model_shorts_runtime.fit(X_train_s, y_train_s)

model_longs_runtime = xgb.XGBRegressor(
    n_estimators=1200, learning_rate=0.015, max_depth=7, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.6, gamma=0.3, objective='reg:squarederror',
    random_state=42, n_jobs=-1
)
model_longs_runtime.fit(X_train_l, y_train_l)

y_pred_orig_s = model_shorts_runtime.predict(X_test_s)
y_pred_orig_l = model_longs_runtime.predict(X_test_l)

y_test_all = np.concatenate([y_test_s, y_test_l])
y_pred_all = np.concatenate([y_pred_orig_s, y_pred_orig_l])

r2 = r2_score(y_test_all, y_pred_all)
rmse = np.sqrt(mean_squared_error(y_test_all, y_pred_all))
mae = mean_absolute_error(y_test_all, y_pred_all)
mape = mean_absolute_percentage_error(y_test_all, y_pred_all)

print(f"\n--- Avaliação Combinada do Pipeline Especialista ---")
print(f"R² Score Global: {r2:.4f}")
print(f"RMSE Global: {rmse:,.2f} minutos")
print(f"MAE Global: {mae:,.2f} minutos")
print(f"MAPE Global: {mape * 100:.2f}%")

print("\n--- Importância das Features - XGBoost Curtas (Top 20) ---")
importances_shorts = sorted(zip(features_shorts, model_shorts_runtime.feature_importances_), key=lambda x: x[1], reverse=True)
for feat, imp in importances_shorts[:20]:
    print(f"{feat}: {imp:.4f}")

print("\n--- Importância das Features - XGBoost Longas (Top 20) ---")
importances_longs = sorted(zip(features_longs, model_longs_runtime.feature_importances_), key=lambda x: x[1], reverse=True)
for feat, imp in importances_longs[:20]:
    print(f"{feat}: {imp:.4f}")

with gzip.open('models/xgb_shorts_runtime_model.pkl.gz', 'wb') as f:
    pickle.dump(model_shorts_runtime, f)

with gzip.open('models/xgb_longs_runtime_model.pkl.gz', 'wb') as f:
    pickle.dump(model_longs_runtime, f)
    
with open('models/features_runtime_model.pkl', 'wb') as f:
    pickle.dump(features_longs, f)


c:\Users\dhavi\OneDrive\Área de Trabalho\Dhavi\UFRPE\3° Período\Projeto Interdisciplinar para Sistemas de Informação III\Projeto_PISI3_2026\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- Avaliação Combinada do Pipeline Especialista ---
R² Score Global: 0.7957
RMSE Global: 19.82 minutos
MAE Global: 12.55 minutos
MAPE Global: 43.68%

--- Importância das Features - XGBoost Curtas (Top 20) ---
genres_Documentary: 0.0966
genres_Animation: 0.0759
is_en: 0.0348
overview_len: 0.0345
genre_mean: 0.0319
genres_TV Movie: 0.0284
genres_Family: 0.0246
genre_decade_mean: 0.0239
production_countries_Japan: 0.0212
production_companies_None: 0.0211
movie_age: 0.0170
genres_Drama: 0.0166
production_countries_Russia: 0.0154
production_countries_Canada: 0.0144
production_companies_Nikkatsu Corporation: 0.0143
production_companies_BBC: 0.0142
has_budget: 0.0139
production_countries_South Korea: 0.0138
release_year: 0.0117
release_5_years: 0.0116

--- Importância das Features - XGBoost Longas (Top 20) ---
production_countries_India: 0.4992
genres_Documentary: 0.0285
genre_mean: 0.0195
genres_Drama: 0.0158
is_en: 0.0157
genres_None: 0.0130
genres_History: 0.0109
production_countries_Phi

In [3]:
import shap
import pickle
import numpy as np
import pandas as pd

explainer_l = shap.TreeExplainer(model_longs_runtime)
X_test_sampled = X_test_l.sample(n=500, random_state=42)
shap_values_l = explainer_l(X_test_sampled)

shap_data_xgb_rt = {
    'features': features_longs,
    'shap_values': shap_values_l.values,
    'feature_values': X_test_sampled.values
}

with open('models/shap/xgb_runtime_shap_data.pkl', 'wb') as f:
    pickle.dump(shap_data_xgb_rt, f)